# 🚀 Automated Parallel PSA Generator Pipeline

This notebook runs the complete 4-way parallel PSA generator pipeline (English -> Swahili, Somali, Luo) on Google Colab's GPU runtime.

### Step 0: Clone Repository
Clone the repository to fetch the required generator scripts and templates, then change into the project directory. If the directory already exists, it will pull the latest updates.

In [ ]:
import os
%cd /content
if not os.path.exists('public-service-anouncement-MT'):
    !git clone https://github.com/SamAbr/public-service-anouncement-MT.git
    %cd public-service-anouncement-MT
else:
    print('Repository directory already exists. Resetting and pulling latest updates...')
    %cd public-service-anouncement-MT
    !git reset --hard HEAD
    !git pull

### Step 1: Install Dependencies
Install the required translation, tracking, and language ID libraries.

In [ ]:
!pip install transformers sentencepiece tqdm pandas torch fasttext nltk gspread gspread-dataframe sentence-transformers

### Step 2: Generate English PSAs & View Online
Generate 50,000 unique, validated, and tag-cohered English announcements. 

Below, you can use Colab's **Interactive Data Table** to search, paginate, and review the entire dataset directly in your browser without downloading anything.

In [ ]:
# Specify your generation size (e.g., 100 for pilot, 5000, or 50000 for final dataset)
SIZE = 5000

# Paste your Azure AI Foundry API key here to use the GPT model
AZURE_API_KEY = "YOUR_AZURE_AI_FOUNDRY_API_KEY"

if AZURE_API_KEY and AZURE_API_KEY != "YOUR_AZURE_AI_FOUNDRY_API_KEY":
    print(f"Running with Azure LLM engine (Size: {SIZE})...")
    !python generate_english_only.py --size {SIZE} --output output/english_psas.csv --engine azure_llm --azure-key {AZURE_API_KEY}
else:
    print(f"Running with offline Template engine (Size: {SIZE})...")
    !python generate_english_only.py --size {SIZE} --output output/english_psas.csv

# Enable Colab's interactive data table view
from google.colab import data_table
import pandas as pd

data_table.enable_dataframe_formatter()
df_review = pd.read_csv('output/english_psas.csv')
df_review

#### Option 2B (Optional): Export to Google Sheets
If you prefer to review the data in Google Sheets instead of the interactive table above, run this cell. It will authenticate with your Google Drive, create a new Google Sheet, and print a shareable link.

In [ ]:
from google.colab import auth
from google.auth import default
import gspread
from gspread_dataframe import set_with_dataframe

print("Authenticating Google account...")
auth.authenticate_user()
creds, _ = default()
gc_client = gspread.authorize(creds)

sheet_name = 'PSA English Seed Dataset for Review'
print(f"Creating new Google Sheet: '{sheet_name}'...")
sh = gc_client.create(sheet_name)
worksheet = sh.get_worksheet(0)

print("Uploading 50,000 records to Google Sheets (this may take 10-15 seconds)...")
set_with_dataframe(worksheet, df_review)

print(f"\nSUCCESS! You can open and review the entire dataset at this link:\n{sh.url}")

### Step 3: Run Sequential Translation Pipeline
Translate the English seed set to Swahili, Somali, and Luo sequentially on the GPU with quality filters (LangID and ChrF round-trip).

**Note:** Proceed to run this step *after* you have reviewed and approved the English dataset.

In [ ]:
!python translate_colab.py --input output/english_psas.csv --output output/psa_parallel_dataset.csv --batch-size 128

### Step 4: Download Parallel Dataset
Download the final parallel CSV file to your local computer.

In [ ]:
from google.colab import files
files.download('output/psa_parallel_dataset.csv')

### Step 5: Save, Commit, and Push to GitHub
Upload the generated CSV file directly to your GitHub repository. 

**Note:** You will need to enter your GitHub Personal Access Token (PAT) securely when prompted.

In [ ]:
import os
import getpass

# Ensure we are in the repository directory
%cd /content/public-service-anouncement-MT

# 1. Prompt for authentication details securely
git_name = input("Enter your Git username: ")
git_email = input("Enter your Git email: ")
git_token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")

# 2. Configure Git in the VM
!git config --global user.name "{git_name}"
!git config --global user.email "{git_email}"

# 3. Configure the push URL with the access token
!git remote set-url origin https://{git_token}@github.com/SamAbr/public-service-anouncement-MT.git

# 4. Stage, commit, and push the CSV file
!git add output/psa_parallel_dataset.csv output/english_psas.csv
!git commit -m "Upload generated parallel datasets from Colab GPU run"
# Force push to main/master depending on active branch
!git push origin HEAD:main --force